# Class 2 — Data Exploration and First Features

**Deliverable:** Feature set v1
**Date:** July 5, 2026

Goal: explore the data more deeply and build the first features. A feature is a clue
engineered from the raw columns that might help separate fraud from normal.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())

## Step 1 — Comparing fraud vs normal

In [ ]:
normal = df[df['Class'] == 0]
fraud  = df[df['Class'] == 1]

print("Normal mean amount:", normal['Amount'].mean())
print("Fraud  mean amount:", fraud['Amount'].mean())

Recorded result: normal 88.29, fraud 122.21.

Fraud is on average ~38% bigger, but the difference is not large — plenty of normal
transactions are far bigger than the fraud average. Amount alone will not be enough.

## Step 2 — Feature 1: amount z-score

In [ ]:
mean_amount = df['Amount'].mean()
std_amount  = df['Amount'].std()
df['amount_zscore'] = (df['Amount'] - mean_amount) / std_amount

# z = 0 is perfectly average; above 3 is conventionally "very unusual"
print(df['amount_zscore'].describe())

## Step 3 — Testing the z-score rule on its own

In [ ]:
print("Fraud  with z-score > 3:", (df[df['Class'] == 1]['amount_zscore'] > 3).sum())
print("Normal with z-score > 3:", (df[df['Class'] == 0]['amount_zscore'] > 3).sum())

Recorded result: **11 fraud caught out of 492**, **4,065 normal wrongly flagged**.

- Recall = 11/492 = **2.2%** — missed almost all the fraud
- Precision = 11/4,076 = **0.27%** — almost everything flagged was innocent

Why it failed: fraudsters deliberately keep amounts small so they blend in. This
failure is the reason the ML upgrade in Class 4 is necessary — a single-column
threshold has a hard ceiling.

## Step 4 — Feature 2: hour of day

In [ ]:
# Time is seconds since the first transaction, not a clock time
df['hour'] = (df['Time'] / 3600) % 24
print(df['hour'].describe())

## Step 5 — Fraud vs normal by hour

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[df['Class'] == 0]['hour'].hist(bins=24, ax=axes[0])
axes[0].set_title('Normal transactions by hour')
df[df['Class'] == 1]['hour'].hist(bins=24, ax=axes[1], color='crimson')
axes[1].set_title('Fraud transactions by hour')
plt.tight_layout()
plt.show()

## What I learned

- How to engineer a feature from raw columns rather than using them directly
- Z-score as a measure of "unusualness" relative to a distribution
- That a visually interesting chart is not proof a feature is useful — Class 3 tests it properly
- Why precision and recall matter more than accuracy on a 99.83% / 0.17% imbalanced set